## Dictyostelium cells
Tanaka M, Kitanishi-Yumura T, Yumura S. A ‘dynamic adder model’for cell size homeostasis in Dictyostelium cells. Scientific Reports. 2021 Jul 2;11(1):13742.

In [1]:
using LinearAlgebra
using DifferentialEquations, Plots
using BenchmarkTools
using HypothesisTests
default(grid=false,legend=false, size=(400,320), dpi=300, tickfontsize=9, lw=2, lc=:gray, labelfontsize=9, tickdir=:out, fontfamily="Arial", legendfontsize=7, markercolor=:indianred, thickness_scaling=2)

mydir = "../helper/"
include(mydir * "base.jl")
include(mydir * "optimization.jl")
include(mydir * "misc.jl")
include(mydir * "myplot.jl")

plot_output (generic function with 3 methods)

In [2]:
# read in data
import XLSX
data_dir = "../data/Dictyostelium/Surface Area VS Time.xlsm"
xlsm_file = XLSX.readxlsx(data_dir)

sheet = xlsm_file["Sheet1"]
data = sheet[:];

data_L = data[3:end,2:2:end]
Lobs_ = [Float64.(collect(skipmissing(data_L[:, i]))) for i in 1:size(data_L, 2)]
data_t = data[3:end,1:2:end]
tobs_ = [Float64.(collect(skipmissing(data_t[:, i]))) for i in 1:size(data_t, 2)]./60
isdiv_, dtobs_, Llast_ = annotate_exp_traj2(Lobs_, tobs_, cutoff=200.0);


### trajectory

In [3]:
p_traj_obs = plot(; size=(750,300), xlim=(0,8), xtick=collect(0:4:8), ylim=(200,1150), ytick=collect(200:400:1150), bottommargin=-3Plots.mm,  leftmargin=-5Plots.mm, background_color=:transparent) # ,xlabel="Time", ylabel="Cell length"
for i in eachindex(Lobs_)
    plot!(p_traj_obs, tobs_[i], Lobs_[i], lc=:snow2, lw=0.2)
end
i = 45
plot!(p_traj_obs, tobs_[i], Lobs_[i], lc=:gray, lw=.4)
scatter!(p_traj_obs, tobs_[i][isdiv_[i]], Lobs_[i][isdiv_[i]], markercolor=:gray, ms=1, markerstrokewidth=0.2)
p_traj_obs
savefig("../figure/SI figure/amoeba/amoeba_traj.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/amoeba/amoeba_traj.pdf"

### correlation

In [4]:
using StatsBase
hedges = collect(100:100:1300);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
h2_obs = fit(Histogram, (vcat( [Lobs_[i][isdiv_[i]][1:end-1] for i in eachindex(Lobs_) ]...), vcat( [Lobs_[i][isdiv_[i]][2:end] for i in eachindex(Lobs_) ]...) ), (hedges,hedges));

h2_obs_normalize = h2_obs.weights / sum(h2_obs.weights) / dhedges^2

svd_h2_obs = svd(h2_obs_normalize);

ll=@layout [a b]
p_h2_obs = plot(; aspect_ratio=1, framestyle=:box, xlim=(200,hedges[end]), ylim=(200,hedges[end]), 
        xtick=collect(300:400:1300), ytick=collect(300:400:1300) )#, title="P(i,i-1)"
heatmap!( p_h2_obs, hbins, hbins, h2_obs_normalize, clim=(0,1e-5),c = :Blues_9,
colorbar_ticks=()) #,colorbar=true

plot(p_h2_obs, layout = ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
savefig("../figure/SI figure/amoeba/amoeba_correlation.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/amoeba/amoeba_correlation.png"

In [5]:
p_S_result = plot(; xlim=[0.5,10.5], ylim=[-0.1,1.05], ytick=[0,0.5,1], xtick=collect(1:4:13), 
size=(300,300), bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
scatter!( p_S_result, svd_h2_obs.S ./ svd_h2_obs.S[1], markercolor=:gray, ms=4, markershape=:hline,msw=1.5)

savefig("../figure/SI figure/amoeba/amoeba_svd.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/amoeba/amoeba_svd.pdf"

In [6]:
# mutual information
P_xy = h2_obs.weights/sum(h2_obs.weights)
P_y = sum(P_xy, dims=1)
P_x = sum(P_xy, dims=2)
is_nan = .!(isnan.(P_xy .* log.(P_xy./(P_x * P_y))))
mi = P_xy .* log.(P_xy./(P_x * P_y))
sum(mi[is_nan])

0.46896625868046554

### input data distribution

In [7]:
using StatsBase
hedges = collect(300:50:1100);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
hh_obs = fit(Histogram, (vcat( [Lobs_[i] for i in eachindex(Lobs_) ]...), vcat( [Llast_[i] for i in eachindex(Llast_) ]...) ), (hedges,hedges));

hh_obs_normalize = hh_obs.weights / sum(hh_obs.weights) / dhedges^2

svd_hh_obs = svd(hh_obs_normalize);

ll=@layout [a b]
p_hh_obs = plot(; aspect_ratio=1, framestyle=:box, xlim=(300,1100), ylim=(300,1100), xtick=collect(300:400:1100), ytick=collect(300:400:1100))#, title="P(i,i-1)" 
heatmap!( p_hh_obs, hbins, hbins, hh_obs_normalize, clim=(0,2e-5),c = :Blues_9,
colorbar_ticks=(),colorbar=false) 

plot(p_hh_obs, layout = ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
savefig("../figure/SI figure/amoeba/amoeba_datadist.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/amoeba/amoeba_datadist.png"

### inference

In [38]:
bf1 = [sChebyshevT{Float64}(),sChebyshevT{Float64}()]
n_func1 = [2, 2]
l01 = [600.0, 600.0]
Θ1, Σ1 = mgs_gen_bases( [vcat(Lobs_...), vcat(Llast_...)], n_func1, bf1, l01);
Θdiv1 = Θ1[vcat(isdiv_...),:];
Θdiv1 = sum(Θdiv1, dims=1)[:];

history1 = sparse_optimization_sweep(sparse_optimization_fg, Θ1, Θdiv1, vcat(dtobs_...); init_cutoff=0.0, coeffs0=1e-4, EPSILON=1E-7, MAX_ITERS=20);

===== start of epoch 1 =====
itr=1: -612.516070996757
itr=2: -636.6776468938422
itr=3: -642.592208275881
itr=4: -643.2490325075222
itr=5: -643.272349639444
itr=6: -643.2570930216926
itr=7: -643.2568038303672
itr=8: -643.2567506940936
itr=1: -612.516070996757
itr=2: -636.6776468938422
itr=3: -642.592208275881
itr=4: -643.2490325075222
itr=5: -643.272349639444
itr=6: -643.2570930216926
itr=7: -643.2568038303672
itr=8: -643.2567506940936
===== end of epoch 1 =====
error = -1.1384675492977716, nterms = 9, cutoff = 0.0
===== start of epoch 2 =====
itr=1: -612.516070996757
itr=2: -636.6776468938422
itr=3: -642.592208275881
itr=4: -643.2490325075222
itr=5: -643.272349639444
itr=6: -643.2570930216926
itr=7: -643.2568038303672
itr=8: -643.2567506940936
itr=1: -611.6921620386461
itr=2: -637.8128366591659
itr=3: -641.8646749814156
itr=4: -642.5915109971107
itr=5: -642.8017173191556
itr=6: -642.9299121409769
itr=7: -643.0250103454342
itr=8: -643.0936166703082
itr=9: -643.1405887850295
itr=10: -643

In [39]:
save_inference_results([vcat(tobs_...) vcat(Lobs_...) vcat(Llast_...)], vcat(isdiv_...), vcat(dtobs_...), 
    [ [@sprintf "%s" typeof(bf1[i]) for i in 1:length(bf1)] n_func1 l01 ], history1; 
    data_type = "exp_amoeba", learn_type = "learn_memory1", output_dir = "./output/", 
    use_uid=true, save_traj=true
)

### plotting

In [8]:
filepath1 = "./output/exp_amoeba_learn_memory1/24-04-03-T1116/"

traj = readdlm(filepath1 * "traj");
tobs = traj[:,1];
Ls0 = [traj[:,j] for j in 2:2];
Ls1 = [traj[:,j] for j in 2:3];

dtobs = readdlm(filepath1 * "dts")[:];
isdiv = convert( BitVector, readdlm(filepath1 * "isdiv")[:] );


In [9]:
# rebuild inference basis function library

function rebuild_library_params(filepath)
    library_sum = readdlm(filepath * "library");
    bf_string = library_sum[:, 1];
    @show bf_string
    n_func = convert(Vector{Int}, library_sum[:,2]);
    l0 = convert(Vector{Float64}, library_sum[:,3]);
    return n_func, l0
end

n_func1, l01 = rebuild_library_params(filepath1);
bf1 = [sChebyshevT{Float64}(),sChebyshevT{Float64}()]
_, Σ1 = mgs_gen_bases(Ls1, n_func1, bf1, l01);

bf_string = Any["sChebyshevT{Float64}", "sChebyshevT{Float64}"]


In [10]:
# read in inference result
function rebuild_history(filepath)

    history_coeffs = readdlm(filepath * "history_coeffs");
    cache_summary  = readdlm(filepath * "history_summary");
    keynames = convert(Vector{String}, cache_summary[1,:] );
    history_metric = cache_summary[2:end,:];
    history = Dict(zip(Symbol.(keynames), [history_metric[:,i] for i = 1:length(keynames)]));

    return history, history_coeffs

end

history1, history_coeffs1 = rebuild_history(filepath1);

In [33]:
# _, imax0 = findmax( -history0[:error] .- 0.5*history0[:logdetH]/sum(dtobs) )
# _, imax1 = findmax( -history1[:error] .- 0.5*history1[:logdetH]/sum(dtobs) )
_, imax1 = findmax(-history1[:error] .- 0.5*compute_logdet.(history1[:H])/sum(vcat(dtobs_...)))

(1.0934457312472412, 2)

In [12]:
# find the maximum BIC score
_, imax1 = findmax(-history1[:error] .- 0.5*history1[:logdetH]/sum(vcat(dtobs_...)))

BIC = -history1[:error] .- 0.5*history1[:logdetH]/sum(dtobs)
plot(history1[:nterms], BIC, lw=2, lc=:dodgerblue4,
    xtick=[0,5,10,15],ytick=[-1,0,1],ylim=(-1.1,1.2), size=(330,300), bottommargin=-6Plots.mm, leftmargin=-3Plots.mm,
    background_color=:transparent)
scatter!([history1[:nterms][imax1]], [BIC[imax1]], ms=2.5, mc=:dodgerblue4, shape=:rect)
savefig("../figure/SI figure/amoeba/amoeba_BIC.pdf")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/amoeba/amoeba_BIC.pdf"

In [16]:
# calculate the inferred λ
ll = collect(300.0:10:1100.0);
ll_ = collect(300.0:10:1100.0);
logλ_learn1 = mgs_rec_bases([ll, ll_], Σ1, history_coeffs1[:, imax1], bf1, l01);

pl1 = plot(;size=(300,300), bottommargin=-6Plots.mm, leftmargin=-3Plots.mm, background_color=:transparent);
heatmap!(pl1, ll, ll_, exp.(logλ_learn1'), xlim=(300,1100), ylim=(300,1100), 
clim=(0,6.0), aspect_ratio=1, framestyle=:box, 
c=:lapaz, xtick=[300,700,1100],ytick=[300,700,1100])
savefig("../figure/SI figure/amoeba/amoeba_memory1.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/amoeba/amoeba_memory1.png"

In [17]:
heatmap(reshape(history_coeffs1[:, imax1],3,3), c =:broc, ticks=false, axis=true,aspect_ratio =1, clim=(-5,5),
xlim=(0.5,3.5), ylim=(0.5,3.5),framestyle=:box,size=(300,300), background_color=:transparent, yflip = true
)
savefig("../figure/SI figure/amoeba/amoeba_coeffs_matrix.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/amoeba/amoeba_coeffs_matrix.png"

### Growth function g inference

In [22]:
# interpolation to smooth the growth function g
function interpolation(t, L, deg)
    A = ones(length(L), deg);
    for i = 2:deg
        A[:,i] = t.^(i-1);
    end
    coeff = A\L;
    D = zeros(length(L), deg);
    for i = 2:deg
        D[:,i] = (i-1) * t.^(i-2);
    end
    return coeff, D * coeff, A * coeff
end

interpolation (generic function with 1 method)

In [23]:
bf_g = Polynomial()
Lt_g = []
Diff_g = []
for Li in eachindex(Lobs_)
    pos = findall(isdiv_[Li]);
    for pi = 1:length(pos)-1
        _,diff, Lt = interpolation(tobs_[Li][pos[pi]+1:pos[pi+1]], Lobs_[Li][pos[pi]+1:pos[pi+1]], 5);
        push!(Lt_g, Lt);
        push!(Diff_g, diff);
    end
end

# linear regression to learn growth rate function g
Lt_g = convert(Vector{Vector{Float64}}, Lt_g);
Diff_g = convert(Vector{Vector{Float64}}, Diff_g);
Lt_g = vcat( [ Lt_g[i]  for i in eachindex(Lt_g)]... );
Diff_g = vcat( [Diff_g[i] for i in eachindex(Diff_g) ]... );
Theta_g = gen_bases( Lt_g,  2);
coeffs_g = Theta_g[:,[1,2]]  \ Diff_g

2-element Vector{Float64}:
 -139.3960757282947
    0.4348437806897866

In [24]:
histogram2d(Lt_g, Diff_g, framestyle=:box, normalize=:pdf, show_empty=true, size=(300,300), background_color=:transparent, bottommargin=-6Plots.mm, leftmargin=-10Plots.mm,
xlim = (250,800), ylims = (-50,250), xtick =[300, 500, 700], ytick=[0, 100, 200], bins=(250:50:800, -50:25:250), c=cgrad(:tempo, rev=false), clim = (0,0.0001)) # bins=(2:0.6:8, 0:3:30),aspect_ratio=0.2, c=:lapaz,
plot!([250,800], coeffs_g[1] .+ coeffs_g[2]*[250,800], lw=1.0, lc=:black, ls=:dash)
savefig("../figure/SI figure/amoeba/amoeba_infered_g.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/amoeba/amoeba_infered_g.png"

### Cut size function h inference

In [20]:
bf_h = Polynomial()
Y_h = vcat( [Lobs_[i][isdiv_[i]] .- Lobs_[i][findall(isdiv_[i]) .+  1]  for i in eachindex(Lobs_)]... );
Theta_h = gen_bases( vcat( [Lobs_[i][isdiv_[i]] for i in eachindex(Lobs_) ]... ), 1);

coeffs_h = Theta_h\ Y_h
coeffs_h[abs.(coeffs_h) .< 0.02] .= 0.0

0-element view(::Vector{Float64}, Int64[]) with eltype Float64

In [21]:
histogram2d( vcat( [Lobs_[i][isdiv_[i]] for i in eachindex(Lobs_) ]... ), Y_h, framestyle=:box, normalize=:pdf, show_empty=true, size=(300,300), background_color=:transparent, bottommargin=-6Plots.mm, leftmargin=-10Plots.mm,
xlim = (500,1100), ylims = (200,600), xtick =[500, 800, 1100], ytick=[200, 400, 600], bins=(500:50:1100, 200:25:600), c=cgrad(:tempo, rev=false),clim=(0,0.0001)) # bins=(2:0.6:8, 0:3:30),aspect_ratio=0.2, c=:lapaz,)
plot!([500,1200], coeffs_h[1] .+ coeffs_h[2] * [500,1200].^1, label="learned", lw=1.0, lc=:black, ls=:dash)
savefig("../figure/SI figure/amoeba/amoeba_infered_h.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/amoeba/amoeba_infered_h.png"

### Resimulation

In [25]:
function f(du, u, p, t)
    bf_f = p[1]
    coeffs_f = p[2]
    du[1] = 0.0
    for n = 1:length(coeffs_f)
        du[1] += coeffs_f[n]*bf_f(n-1, u[1])
    end
end

function rate_memory(u, p, t)
    Σ1 = p[5]
    coeffs = p[6]
    bf1 = p[7]
    l01 = p[8]
    ln_rate = mgs_rec_bases([[u[1]], [u[2]]], Σ1, coeffs, bf1, l01);
    return exp(ln_rate[1])
end

function affect!(integrator)
    integrator.u[2] = integrator.u[1]
    bf_h = integrator.p[3]
    coeffs_h = integrator.p[4]
    u_old = integrator.u[1]
    for n = 1:length(coeffs_h)
        integrator.u[1] -= coeffs_h[n] * bf_h(n-1, u_old)
    end
end

affect! (generic function with 1 method)

In [ ]:
# resimulate data and save or read in your resimulated data
# resimulate data based on your inferred parameters
Lresult = []
tresult = []
for i in eachindex(Lobs_)
    vrj = VariableRateJump(rate_memory, affect!)
    oprob = ODEProblem(f, [Lobs_[i][1];2 * Lobs_[i][1]], (0.0,100*maximum(tobs_[i])), (bf_g, coeffs_g, bf_h, coeffs_h, Σ1, history_coeffs1[:, imax1], bf1, l01))
    jprob = JumpProblem(oprob, Direct(), vrj)
    sol = solve(jprob,Tsit5(),dtmax=0.01);
    L_r = Array(sol')[:,1];
    t_r = sol.t;
    push!(Lresult, L_r);
    push!(tresult, t_r);
end

Lresult = convert(Vector{Vector{Float64}}, Lresult);
tresult = convert(Vector{Vector{Float64}}, tresult);

isdiv_result = []
for i in eachindex(Lresult)
    isdiv_r, _ = annotate_sim_traj(Lresult[i], tresult[i])
    push!(isdiv_result, isdiv_r);
end

writedlm("../figure/SI figure/amoeba/amoeba_resim_data.txt", [Lresult, tresult], '\t')

In [31]:
# read in resimulated data

using JSON
resim_data = readdlm("../figure/SI figure/amoeba/amoeba_resim_data.txt", '\t')

Lresult = [Float64.(JSON.parse(resim_data[1, i])) for i in 1:size(resim_data,2)];
tresult = [Float64.(JSON.parse(resim_data[2, i])) for i in 1:size(resim_data,2)];

isdiv_result = []
for i in eachindex(Lresult)
    isdiv_r, _ = annotate_sim_traj(Lresult[i], tresult[i])
    push!(isdiv_result, isdiv_r);
end

In [33]:
p_traj = plot(; size=(750,300),xlim=(0,8), xtick=collect(0:4:8), ylim=(200,1150), ytick=collect(200:400:1150), bottommargin=-3Plots.mm,  leftmargin=-5Plots.mm, background_color=:transparent)
for i =1:50
    plot!(p_traj, tresult[i], Lresult[i], lc=:snow2, lw=0.2)
end
plot!(p_traj, tresult[3], Lresult[3], lw = 0.4, lc=:seagreen)
scatter!(p_traj, tresult[3][isdiv_result[3]], Lresult[3][isdiv_result[3]], markercolor=:seagreen, ms=1, markerstrokewidth=0.2)
savefig("../figure/SI figure/amoeba/amoeba_resim.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/amoeba/amoeba_resim.pdf"

In [34]:
using StatsBase
hedges = collect(100:100:1300);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
h2_result = fit(Histogram, (vcat( [Lresult[i][isdiv_result[i]][1:end-1] for i in eachindex(Lresult) ]...), vcat( [Lresult[i][isdiv_result[i]][2:end] for i in eachindex(Lresult) ]...) ), (hedges,hedges));

h2_result_normalize = h2_result.weights / sum(h2_result.weights) / dhedges^2

svd_h2_result = svd(h2_result_normalize);

ll=@layout [a b]
p_h2_result = plot(; aspect_ratio=1, framestyle=:box, xlim=(200,hedges[end]), ylim=(200,hedges[end]), 
        xtick=collect(300:400:1300), ytick=collect(300:400:1300) )#, title="P(i,i-1)"
heatmap!( p_h2_result, hbins, hbins, h2_result_normalize, clim=(0,1e-5),c = :Blues_9,
colorbar_ticks=()) #,colorbar=true

plot(p_h2_result, layout = ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
savefig("../figure/SI figure/amoeba/amoeba_correlation_resim.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/amoeba/amoeba_correlation_resim.png"

In [35]:
p_S_result = plot(; xlim=[0.5,10.5], ylim=[-0.1,1.05], ytick=[0,0.5,1], xtick=collect(1:4:13), 
size=(300,300), bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
scatter!( p_S_result, svd_h2_result.S ./ svd_h2_result.S[1], markercolor=:seagreen, ms=3.5, markershape=:circle,msw=1, msc=:seagreen)
scatter!( p_S_result, svd_h2_obs.S ./ svd_h2_obs.S[1], markercolor=:gray, ms=4, markershape=:hline,msw=3)
savefig("../figure/SI figure/amoeba/amoeba_svd_resim.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/amoeba/amoeba_svd_resim.pdf"

In [36]:
# mutual information
P_xy = h2_result.weights/sum(h2_result.weights)
P_y = sum(P_xy, dims=1)
P_x = sum(P_xy, dims=2)
is_nan = .!(isnan.(P_xy .* log.(P_xy./(P_x * P_y))))
mi = P_xy .* log.(P_xy./(P_x * P_y))
sum(mi[is_nan])

0.2798558485544879

### histogram

In [38]:
p1 = plot(;xlim=(400,1300),  xtick=collect(400:400:1200),ylim=(-0.0002,0.005),ytick=[0,0.002,0.004])
stephist!( p1, vcat( [Lobs_[i][isdiv_[i]] for i in eachindex(Lobs_) ]...), bins=collect(400:120:1300), normalize=:pdf, lw=0.2#, xlabel="Cell length", ylabel="PDF"
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)

stephist!( p1, vcat( [Lresult[i][isdiv_result[i]] for i in eachindex(Lresult) ]...), normalize=:pdf, bins=collect(400:120:1300), lc=:seagreen, lw=0.2)
savefig("../figure/SI figure/amoeba/amoeba_histogram_divsize.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/amoeba/amoeba_histogram_divsize.pdf"

In [39]:
p2 = plot(;xlim=(0,10.02), xtick=collect(0:5:10),ylim=(-0.02,0.32),ytick=[0,.1,.2,.3])
stephist!( p2, vcat( diff.([tobs_[i][isdiv_[i]] for i in eachindex(tobs_)]) ... ), normalize=:pdf
        , bins=collect(0:1:10), lw=0.2
        , bottommargin=-6Plots.mm, leftmargin=0Plots.mm, background_color=:transparent)
        #,ylabel="PDF", xlabel="Generation time" )
stephist!( p2, vcat( diff.([tresult[i][isdiv_result[i]] for i in eachindex(tresult)]) ... ), normalize=:pdf
, lc=:seagreen, bins=collect(0:1:10), lw=0.2)

savefig("../figure/SI figure/amoeba/amoeba_histogram_gentime.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/amoeba/amoeba_histogram_gentime.pdf"